# DTD Captions: Qwen3-VL-8B-Instruct vs Qwen3-VL-32B-Instruct

Generate two new full-dataset caption CSVs, then train the existing **M-TACS notebook 05 full_both**
separately with each CSV. This notebook performs **frozen inference only**, not Qwen fine-tuning or classifier training.
Both methods use the same fixed prompt, image pixel budget, greedy decoding, dtype, seed and `dtd_lexical_v2` policy.
No CLIP image-text reranking, caption retries or class-conditioned repair is added. This changes the captioner,
not the number of selected context images: downstream **K remains 1**.

DTD split 1 uses the official `labels/train1.txt`, `val1.txt`, `test1.txt`: 1,880 images each, 5,640 total.
An image-derived caption is generated for every query/candidate image. The VLM receives only its image pixels
and the same English prompt, never an image ID, path, class label, class list, reference caption or test score.
The global 47-class vocabulary is used only AFTER generation for lexical checks and diagnostic CLIP scores.

Run all cells in order. 8B generation and audit finish before 32B starts; each stage runs in its own process,
which releases GPU memory on exit. Per-image atomic caches resume missing work after interruption.
Only complete, audited exports are named `captions.csv`; smoke tests/partial outputs cannot masquerade as complete files.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import json
import sys
from pathlib import Path

print("Offline mode: no model downloads, API calls, pip installs or DataLoader/fork workers.")


In [ ]:
# Set exact local folders when auto-discovery cannot find the attached checkpoints.
DTD_ROOT = Path(os.getenv("DTD_ROOT", "/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd"))
DTD_SPLIT = int(os.getenv("DTD_SPLIT", "1"))
QWEN_8B_MODEL_PATH = os.getenv("QWEN_8B_MODEL_PATH", "")
QWEN_32B_MODEL_PATH = os.getenv("QWEN_32B_MODEL_PATH", "")
CLIP_MODEL_LOCAL_PATH = os.getenv("CLIP_MODEL_LOCAL_PATH", "/kaggle/input/datasets/nguyenducxu/vit-32/model_clip_vit")
SEARCH_ROOTS = [Path("/kaggle/input")]

# Optional attached site-packages folder containing modern Transformers AND compatible dependencies.
# Leave blank when the Kaggle kernel already supports Qwen3-VL. Florence's 4.49 runtime is NOT sufficient.
QWEN_RUNTIME_SITE_PACKAGES = os.getenv("QWEN_RUNTIME_SITE_PACKAGES", "")
PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_ROOT = Path(os.getenv("DTD_QWEN_CAPTION_OUTPUT", str(
    PROJECT_DIR / "outputs" / f"qwen3vl_dtd_split{DTD_SPLIT}_captions_v1")))
RUN_8B = True
RUN_32B = True
MAX_NEW_IMAGES = 0  # 0 = full dataset; set 8 for a smoke test, then reset to 0 and rerun generation cells.

SETTINGS = {
    "batch_size": 1,  # Same conservative batch for BOTH models; increase before starting a new experiment.
    "dtype": "bfloat16",  # Use float16 for BOTH models if the GPU does not support BF16.
    "min_pixels": 256 * 256,
    "max_pixels": 512 * 512,
    "max_new_tokens": 48,
    "seed": 42,
    "audit_batch_size": 128,
    "semantic_threshold": 0.30,  # Diagnostic only, never a hard mask in dtd_lexical_v2.
}
METHOD_OUTPUTS = {
    "qwen3_vl_8b_instruct": OUTPUT_ROOT / "qwen3_vl_8b_instruct",
    "qwen3_vl_32b_instruct": OUTPUT_ROOT / "qwen3_vl_32b_instruct",
}
print("Output root:", OUTPUT_ROOT)


## Required Offline Inputs and Hardware

Attach the **complete original floating-point Instruct** folders for:
- `Qwen/Qwen3-VL-8B-Instruct`
- `Qwen/Qwen3-VL-32B-Instruct`
- The same full CLIP ViT-B/32 checkpoint used by your DTD ablations, and the DTD dataset.

Set `QWEN_8B_MODEL_PATH` / `QWEN_32B_MODEL_PATH` to the folder containing `config.json`, all safetensors shards
and their index, tokenizer files, chat template, processor and image-preprocessor configuration. Automatic discovery
accepts only an unambiguous folder named for the requested Instruct model. Size is verified from safetensors headers;
the config alone cannot prove Instruct vs Thinking, so attach the correct official repository. All asset contents,
including templates and weights, are hashed into the saved protocol. Large-model hashing can take several minutes.

The worker needs PyTorch/torchvision, Pillow, numpy, tqdm, accelerate and **Transformers with Qwen3-VL support
(>=4.57)**. A compatible current Kaggle 5.x installation can be used. The original offline Florence bundle's
Transformers 4.49 wheels do NOT support Qwen3-VL. No package changes are made automatically. With Internet disabled,
attach compatible offline dependencies beforehand or set `QWEN_RUNTIME_SITE_PACKAGES` to a prepared runtime folder.
Do not point it at the legacy Florence runtime or unzip individual packages without their compatible dependencies.

Default inference is BF16 + SDPA, batch 1, no quantization, CPU offload or FlashAttention installation.
32B weights alone require roughly 60+ GiB at 16-bit precision; activations, image tokens and generation cache need
additional memory. An 80/96 GiB-class GPU is the intended starting point, not a guarantee of fitting every setting.
8B is lighter but a 16 GiB GPU can still be insufficient. Free other GPU models before Run All.
Changing dtype/resolution/prompt/settings/assets/runtime requires a NEW output root; do not mix protocols on resume.


In [ ]:
"""Offline, resumable Qwen3-VL caption exports for the existing DTD ablations."""
from __future__ import annotations

import csv
from collections import defaultdict
from pathlib import PurePosixPath
import gc
import importlib.metadata
import json
import math
import os
import re
import struct
import subprocess
import sys
import time
from pathlib import Path

def atomic_csv(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)

def checkpoint_asset_identity(model_dir):
    model_dir = Path(model_dir)
    files = sorted(path for path in model_dir.iterdir() if path.is_file() and
                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))
    if not any(path.suffix in {".safetensors", ".bin"} for path in files):
        raise ValueError("Checkpoint has no weight files")
    return {path.name: file_sha256(path) for path in files}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def load_caption_manifest(root, split, expected_classes=47, per_class=40):
    root = Path(root).expanduser().resolve()
    if not 1 <= int(split) <= 10:
        raise ValueError("DTD_SPLIT must be in 1..10")
    images = (root / "images").resolve()
    names = sorted(path.name for path in images.iterdir() if path.is_dir())
    if len(names) != expected_classes:
        raise ValueError(f"Expected {expected_classes} DTD image folders; found {len(names)} at {images}")
    records, seen = [], set()
    for role in ("train", "val", "test"):
        counts = defaultdict(int)
        for raw in (root / "labels" / f"{role}{split}.txt").read_text(encoding="utf-8-sig").splitlines():
            text = raw.strip().replace("\\", "/")
            if not text:
                continue
            parts = PurePosixPath(text).parts
            if len(parts) != 2 or parts[0] not in names or any(part in {"..", "."} for part in parts):
                raise ValueError(f"Unexpected official DTD path: {text!r}")
            image_id = PurePosixPath(*parts).as_posix()
            path = images.joinpath(*parts).resolve()
            if not path.is_relative_to(images) or not path.is_file():
                raise FileNotFoundError(f"DTD image missing/outside images root: {path}")
            if image_id in seen:
                raise ValueError(f"Duplicate image or overlapping official splits: {image_id}")
            seen.add(image_id)
            counts[parts[0]] += 1
            records.append({"image_id": image_id, "image_path": str(path), "split": role})
        if set(counts) != set(names) or any(count != per_class for count in counts.values()):
            raise ValueError(f"Official {role}{split}.txt must contain {per_class} images/class")
    return records, names

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,
                                    allow_nan=False).encode()).hexdigest()

def read_csv_rows(path, required=()):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):
            raise ValueError(f"Missing/duplicate CSV header: {path}")
        if not set(required).issubset(reader.fieldnames):
            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")
        rows = list(reader)
    if any(None in row or any(value is None for value in row.values()) for row in rows):
        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")
    return rows
def clean_dtd_caption(text):
    import re
    text = str(text).strip()
    for _ in range(3):
        before = text
        text = re.sub(r"^(?:(?:the|this)\s+)?(?:image|picture|photo(?:graph)?)\s+"
                      r"(?:is|shows|depicts|features|contains)\s+", "", text, flags=re.I)
        text = re.sub(r"^(?:(?:a|an|the)\s+)?close[ -]up(?:\s+view)?\s+(?:of|showing)\s+",
                      "", text, flags=re.I)
        text = re.sub(r"^(?:a|an|the)(?:\s+|$)", "", text, flags=re.I)
        if text == before:
            break
    words = re.findall(r"[A-Za-z0-9]+(?:['\u2019][A-Za-z0-9]+)?", text)
    selected = []
    normalized_count = 0
    for word in words:
        count = len(re.findall(r"[A-Za-z0-9]+", word))
        if normalized_count + count > 8:
            break
        selected.append(word)
        normalized_count += count
    return " ".join(selected)
ALIASES = {
    "banded": ["bands", "banding"], "blotchy": ["blotches", "blotch"], "braided": ["braid", "braids"],
    "bubbly": ["bubbles", "bubble"], "bumpy": ["bumps", "bump"], "chequered": ["checkered", "checkerboard"],
    "cobwebbed": ["cobwebs", "cobweb"], "cracked": ["cracks", "crack"],
    "crosshatched": ["crosshatched", "cross hatching"], "crystalline": ["crystals", "crystal"],
    "dotted": ["dots", "polka dots"], "fibrous": ["fibers", "fibres"], "flecked": ["flecks"],
    "freckled": ["freckles"], "frilly": ["frills"], "gauzy": ["gauze"], "gridded": ["grid", "grids"],
    "grooved": ["grooves"], "honeycombed": ["honeycomb"], "interlaced": ["interlacing"],
    "knitted": ["knit", "knitting"], "lacelike": ["lace", "lace like"], "lined": ["parallel lines"],
    "marbled": ["marbling"], "meshed": ["mesh"], "paisley": [], "perforated": ["perforations"],
    "pitted": ["pits"], "pleated": ["pleats"], "polka-dotted": ["polka dotted", "polka dots"],
    "porous": ["pores"], "potholed": ["potholes"], "scaly": ["scales"], "smeared": ["smears"],
    "spiralled": ["spiraled", "spirals", "spiral"], "sprinkled": ["sprinkles"], "stained": ["stains"],
    "stratified": ["strata"], "striped": ["stripes", "stripe"], "studded": ["studs"],
    "swirly": ["swirls", "swirl"], "veined": ["veins"], "waffled": ["waffle"],
    "woven": ["weave", "weaving"], "wrinkled": ["wrinkles"], "zigzagged": ["zigzag", "zig zag"],
}

def bool_value(value):
    return str(value).lower() in {"true", "yes", "1"}

def load_rgb(path):
    from PIL import Image, ImageOps
    with Image.open(path) as source:
        return ImageOps.exif_transpose(source).convert("RGB")

def normalize(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).replace("_", " ").lower()).strip()

def term_hits(text, names, aliases=False):
    padded = " " + normalize(text) + " "
    terms = set(normalize(name) for name in names)
    if aliases:
        terms.update(normalize(term) for name in names for term in ALIASES.get(name, []))
    return sorted(term for term in terms if term and " " + term + " " in padded)

def valid_short(text, names):
    return 5 <= len(normalize(text).split()) <= 8 and not term_hits(text, names)


METHOD_VERSION = "qwen3vl_dtd_short_caption_v1"
MODEL_IDS = {
    "qwen3_vl_8b_instruct": "Qwen/Qwen3-VL-8B-Instruct",
    "qwen3_vl_32b_instruct": "Qwen/Qwen3-VL-32B-Instruct",
}
CAPTION_PROMPT = (
    "Describe only the visible surface texture in 5 to 8 English words. "
    "Focus on visible colors, local shapes, spacing, repetition, and surface structure. "
    "Do not identify the texture category or guess the object, use, location, or hidden material. "
    "Start directly with descriptive words, not an article or an image introduction. "
    "Output only one short description, without a heading, explanation, or quotation marks."
)
COLUMNS = [
    "image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",
    "lexical_leakage_detected", "semantic_leakage_detected", "semantic_leakage_score",
    "semantic_leakage_class", "retry_count", "vlm_model", "prompt_version",
    "caption_input_policy", "semantic_audit_status", "captioner_fingerprint", "generation_method",
    "source_long_caption", "source_word_count", "word_count", "cleanup_changed_text",
    "alias_hit_diagnostic", "generation_token_limit_reached", "effective_batch_size", "audit_fingerprint",
]


def local_model_configs(roots):
    seen = set()
    for root in roots:
        root = Path(root).expanduser()
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {
                "images", ".git", "__pycache__", "node_modules", ".cache"})
            if len(Path(current).relative_to(root).parts) >= 9:
                dirs[:] = []
            if "config.json" not in files:
                continue
            path = Path(current).resolve()
            if str(path) not in seen:
                seen.add(str(path))
                yield path


def weight_files(path):
    path = Path(path).resolve()
    index = path / "model.safetensors.index.json"
    if index.is_file():
        names = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())
        if not names:
            raise ValueError(f"Empty safetensors weight index: {index}")
        if any(not isinstance(name, str) or "/" in name or "\\" in name or Path(name).is_absolute() for name in names):
            raise ValueError(f"Weight index must reference shard filenames inside its model folder: {index}")
        files = [path / name for name in sorted(names)]
    else:
        files = [path / "model.safetensors"]
    for file in files:
        # HF snapshot files may be read-only symlinks into a sibling blobs directory.
        if not file.is_file() or file.suffix != ".safetensors":
            raise FileNotFoundError(f"Missing/unsafe local safetensors shard: {file}")
    return files


def safetensors_parameter_count(files):
    # Read headers only, not tensor contents; verify the attached model's actual scale.
    count = 0
    for path in files:
        with Path(path).open("rb") as handle:
            raw_length = handle.read(8)
            if len(raw_length) != 8:
                raise ValueError(f"Truncated safetensors header: {path}")
            length = struct.unpack("<Q", raw_length)[0]
            if not 2 <= length <= 64 * 1024 * 1024 or length + 8 > Path(path).stat().st_size:
                raise ValueError(f"Invalid safetensors header length: {path}")
            header = json.loads(handle.read(length))
        for name, tensor in header.items():
            if name == "__metadata__":
                continue
            if tensor.get("dtype") not in {"F16", "BF16", "F32"}:
                raise ValueError("Use original floating-point weights, not quantized/FP8 weights, for this comparison")
            shape = tensor.get("shape", [])
            if not all(type(size) is int and size >= 0 for size in shape):
                raise ValueError(f"Invalid tensor shape: {path}: {name}")
            count += math.prod(shape)
    return count


def validate_qwen_folder(path, method):
    path = Path(path).expanduser().resolve()
    if method not in MODEL_IDS:
        raise ValueError(f"Unknown Qwen method: {method}")
    config = json.loads((path / "config.json").read_text(encoding="utf-8"))
    if config.get("model_type") != "qwen3_vl" or config.get("quantization_config"):
        raise ValueError("Expected original dense Qwen3-VL, not Qwen2.5, pure-text Qwen, MoE or a quantized model")
    if "thinking" in str(path).lower():
        raise ValueError("Attach the Instruct checkpoint, not Thinking")
    for name in ("tokenizer_config.json", "preprocessor_config.json"):
        if not (path / name).is_file():
            raise FileNotFoundError(f"Missing local processor/tokenizer asset: {path / name}")
    if not (path / "tokenizer.json").is_file():
        raise FileNotFoundError(f"Missing tokenizer.json in {path}")
    tokenizer = json.loads((path / "tokenizer_config.json").read_text(encoding="utf-8"))
    template_json = path / "chat_template.json"
    json_template = json.loads(template_json.read_text(encoding="utf-8")).get("chat_template") if template_json.is_file() else None
    if not tokenizer.get("chat_template") and not (path / "chat_template.jinja").is_file() and not json_template:
        raise FileNotFoundError(f"Missing Qwen chat template in {path}")
    files = weight_files(path)
    count = safetensors_parameter_count(files)
    low, high = (7_000_000_000, 11_000_000_000) if "8b" in method else (28_000_000_000, 38_000_000_000)
    if not low <= count <= high:
        raise ValueError(f"Wrong checkpoint size for {MODEL_IDS[method]}: {count:,} stored parameters")
    return path, count, sum(file.stat().st_size for file in files)


def resolve_qwen_model(reference, method, roots):
    if str(reference).strip():
        path, _, _ = validate_qwen_folder(reference, method)
        return path
    needle = MODEL_IDS[method].split("/")[-1].lower()
    matches = [path for path in local_model_configs(roots) if needle in str(path).lower()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Set the exact local path for {MODEL_IDS[method]}. Found {len(matches)} named matches. "
            "Attach its complete Hugging Face folder as Kaggle Input; Florence weights are not Qwen weights. "
            "No download is attempted.")
    return validate_qwen_folder(matches[0], method)[0]


def resolve_clip_model(reference, roots):
    def is_clip(path):
        try:
            return json.loads((path / "config.json").read_text(encoding="utf-8")).get("model_type") == "clip"
        except (OSError, ValueError):
            return False
    if str(reference).strip() and Path(reference).is_dir():
        path = Path(reference).resolve()
        if not is_clip(path):
            raise ValueError(f"Not a full local CLIP checkpoint: {path}")
        return path
    matches = [path for path in local_model_configs(roots) if is_clip(path)]
    if len(matches) != 1:
        raise FileNotFoundError("Set CLIP_MODEL_LOCAL_PATH to the exact full CLIP ViT-B/32 folder")
    return matches[0]


def qwen_asset_identity(path):
    result = checkpoint_asset_identity(path)
    for template in sorted(Path(path).glob("*.jinja")):
        result[template.name] = file_sha256(template)
    return result


def fingerprint_records(records):
    return [{"image_id": row["image_id"], "split": row["split"],
             "sha256": file_sha256(row["image_path"])} for row in records]


def runtime_versions():
    result = {}
    for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers", "Pillow", "numpy"):
        result[name] = importlib.metadata.version(name)
    return result


def validate_settings(settings):
    for key in ("batch_size", "max_new_tokens", "min_pixels", "max_pixels", "audit_batch_size"):
        if type(settings[key]) is not int or settings[key] < 1:
            raise ValueError(f"{key} must be a positive integer")
    if settings["min_pixels"] > settings["max_pixels"]:
        raise ValueError("min_pixels must not exceed max_pixels")
    if type(settings["seed"]) is not int or not 0 <= settings["seed"] < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    if settings["dtype"] not in {"bfloat16", "float16"}:
        raise ValueError("Use the same bfloat16 or float16 dtype for both captioners")
    if not math.isfinite(settings["semantic_threshold"]) or not -1 <= settings["semantic_threshold"] <= 1:
        raise ValueError("semantic_threshold must be finite and in [-1, 1]")


def lock_protocol(path, protocol):
    path = Path(path)
    if path.is_file():
        if json.loads(path.read_text(encoding="utf-8")) != protocol:
            raise ValueError(f"Changed images/model/prompt/runtime/settings. Use a new output directory: {path.parent}")
    else:
        if path.parent.exists() and any(path.parent.iterdir()):
            raise ValueError(f"Refusing to adopt cached output without its protocol: {path.parent}")
        atomic_json(path, protocol)
    return object_hash(protocol)


def cache_path(directory, image_id):
    return Path(directory) / (object_hash(image_id) + ".json")


def cached_row(directory, record, identity):
    path = cache_path(directory, record["image_id"])
    if not path.is_file():
        return None
    result = json.loads(path.read_text(encoding="utf-8"))
    if (result.get("identity") != identity or result.get("row", {}).get("image_id") != record["image_id"]
            or result["row"].get("split") != record["split"]):
        raise ValueError(f"Wrong cached image/protocol: {path}")
    return result["row"]


def generation_row(record, response, method, identity, names, token_limit=False, batch_size=1):
    text = clean_dtd_caption(response)
    lexical = bool(term_hits(text, names))
    return {
        "image_id": record["image_id"], "split": record["split"], "raw_caption": text, "final_caption": text,
        "caption_valid": valid_short(text, names), "leakage_detected": lexical,
        "lexical_leakage_detected": lexical, "semantic_leakage_detected": False,
        "semantic_leakage_score": "", "semantic_leakage_class": "", "retry_count": 0,
        "vlm_model": MODEL_IDS[method], "prompt_version": METHOD_VERSION,
        "caption_input_policy": "dtd_lexical_v2", "semantic_audit_status": "pending",
        "captioner_fingerprint": identity, "generation_method": method,
        "source_long_caption": str(response), "source_word_count": len(normalize(response).split()),
        "word_count": len(normalize(text).split()), "cleanup_changed_text": str(response).strip() != text,
        "alias_hit_diagnostic": bool(term_hits(text, names, aliases=True)),
        "generation_token_limit_reached": bool(token_limit), "effective_batch_size": batch_size,
        "audit_fingerprint": "",
    }


def audited_row(row, score, class_name, identity, threshold):
    if not math.isfinite(score) or not -1.00001 <= score <= 1.00001 or not class_name:
        raise ValueError("Invalid CLIP class-audit result; do not fill missing scores with zero")
    result = dict(row)
    result.update(semantic_leakage_score=round(score, 6), semantic_leakage_class=class_name,
                  semantic_leakage_detected=score >= threshold,
                  semantic_audit_status="diagnostic_only", audit_fingerprint=identity)
    # dtd_lexical_v2 deliberately does not hard-reject text using class-CLIP similarity.
    return result


def caption_messages(images, prompt=CAPTION_PROMPT):
    return [{"role": "user", "content": [{"type": "image", "image": image},
            {"type": "text", "text": prompt}]} for image in images]


class FrozenQwenCaptioner:
    def __init__(self, path, settings):
        import torch
        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
        self.device = torch.device("cuda:0")
        self.settings = settings
        dtype = getattr(torch, settings["dtype"])
        if dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():
            raise RuntimeError("This GPU lacks BF16 support. Set dtype=float16 for BOTH models in a new output directory")
        self.processor = AutoProcessor.from_pretrained(str(path), local_files_only=True, trust_remote_code=False,
            min_pixels=settings["min_pixels"], max_pixels=settings["max_pixels"], use_fast=True)
        self.processor.tokenizer.padding_side = "left"
        self.model, info = Qwen3VLForConditionalGeneration.from_pretrained(str(path), local_files_only=True,
            trust_remote_code=False, dtype=dtype, device_map={"": str(self.device)},
            attn_implementation="sdpa", low_cpu_mem_usage=True, output_loading_info=True)
        if info.get("missing_keys") or info.get("mismatched_keys"):
            raise ValueError(f"Incomplete Qwen learned weights: {info}")
        self.model.eval().requires_grad_(False)

    def generate(self, images):
        import torch
        conversations = [[message] for message in caption_messages(images)]
        prompts = self.processor.apply_chat_template(conversations, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=prompts, images=images, padding=True, return_tensors="pt",
            min_pixels=self.settings["min_pixels"], max_pixels=self.settings["max_pixels"])
        inputs.pop("token_type_ids", None)
        inputs = {key: value.to(self.device) if torch.is_tensor(value) else value for key, value in inputs.items()}
        with torch.inference_mode():
            output = self.model.generate(**inputs, max_new_tokens=self.settings["max_new_tokens"],
                do_sample=False, num_beams=1, use_cache=True, temperature=None, top_p=None, top_k=None)
        suffix = output[:, inputs["input_ids"].shape[1]:]
        texts = self.processor.batch_decode(suffix, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        eos = self.model.generation_config.eos_token_id
        eos = set(eos if isinstance(eos, (list, tuple)) else [eos])
        limits = [len(tokens) >= self.settings["max_new_tokens"] and not any(token in eos for token in tokens)
                  for tokens in suffix.cpu().tolist()]
        if len(texts) != len(images):
            raise ValueError("Qwen output count differs from input image count")
        return list(zip(texts, limits))


class ClipClassAuditor:
    def __init__(self, path, names):
        import torch
        from transformers import AutoTokenizer, CLIPTextModelWithProjection
        self.tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
        self.model, info = CLIPTextModelWithProjection.from_pretrained(str(path), local_files_only=True,
            output_loading_info=True)
        missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]
        if missing or info.get("mismatched_keys"):
            raise ValueError(f"Incomplete CLIP text tower/projection: {missing}")
        self.model = self.model.to("cuda:0").eval().requires_grad_(False)
        self.names = names
        prompts = [template.format(name=name.replace("_", " ").replace("-", " ")) for name in names
                   for template in ("a photo of {name}", "a scene showing {name}")]
        self.class_vectors = torch.nn.functional.normalize(self.encode(prompts).reshape(len(names), 2, -1).mean(1), dim=1)

    def encode(self, texts):
        import torch
        tokens = self.tokenizer(list(texts), padding=True, truncation=True,
            max_length=self.model.config.max_position_embeddings, return_tensors="pt")
        tokens = {key: value.to("cuda:0") for key, value in tokens.items()}
        with torch.inference_mode():
            features = self.model(**tokens).text_embeds
        return torch.nn.functional.normalize(features.float(), dim=1)

    def score(self, texts):
        scores, indices = (self.encode(texts) @ self.class_vectors.T).max(dim=1)
        return [(float(score), self.names[index]) for score, index in zip(scores.cpu(), indices.cpu().tolist())]


def available_rows(directory, records, identity):
    return [row for record in records if (row := cached_row(directory, record, identity)) is not None]


def generate_records(payload, identity, captioner_factory=FrozenQwenCaptioner):
    import torch
    from tqdm.auto import tqdm
    output = Path(payload["output"])
    cache = output / "generated"
    records, settings = payload["records"], payload["settings"]
    todo = [row for row in records if cached_row(cache, row, identity) is None]
    cached_count = len(records) - len(todo)
    limit = payload.get("max_new_images", 0)
    if limit:
        todo = todo[:limit]
    started = time.perf_counter()
    print(f"{payload['method']}: cached={cached_count}, queued={len(todo)}", flush=True)
    if todo:
        model = captioner_factory(payload["model_path"], settings)
        current, offset = settings["batch_size"], 0
        progress = tqdm(total=len(todo), desc=payload["method"], unit="image")
        try:
            while offset < len(todo):
                group = todo[offset:offset + current]
                images = []
                try:
                    for row in group:
                        if file_sha256(row["image_path"]) != row["image_sha256"]:
                            raise ValueError(f"Image changed after manifest hashing: {row['image_id']}")
                        images.append(load_rgb(row["image_path"]))
                    responses = model.generate(images)
                    if len(responses) != len(group):
                        raise ValueError("Incorrect caption batch length; batch not committed")
                except torch.cuda.OutOfMemoryError:
                    if current == 1:
                        raise RuntimeError("Qwen OOM at batch 1. Free GPU memory or use a larger GPU; completed images are cached") from None
                    current = max(1, current // 2)
                    gc.collect()
                    torch.cuda.empty_cache()
                    print(f"CUDA OOM: generation batch reduced to {current}; no image skipped", flush=True)
                    continue
                finally:
                    for image in images:
                        image.close()
                for record, (text, token_limit) in zip(group, responses):
                    row = generation_row(record, text, payload["method"], identity,
                        payload["class_names"], token_limit, len(group))
                    atomic_json(cache_path(cache, record["image_id"]), {"identity": identity, "row": row})
                offset += len(group)
                progress.update(len(group))
        finally:
            progress.close()
            del model
            gc.collect()
            torch.cuda.empty_cache()
    rows = available_rows(cache, records, identity)
    atomic_csv(output / "captions_raw_partial.csv", rows, COLUMNS)
    atomic_json(output / "generation_status.json", {"identity": identity, "rows": len(rows),
                "expected": len(records), "complete": len(rows) == len(records),
                "new_images_this_invocation": len(todo), "elapsed_seconds_this_invocation": time.perf_counter() - started})


def audit_records(payload, generation_identity, audit_identity, auditor_factory=ClipClassAuditor):
    from tqdm.auto import tqdm
    output, records = Path(payload["output"]), payload["records"]
    raw_rows = available_rows(output / "generated", records, generation_identity)
    by_id = {row["image_id"]: row for row in raw_rows}
    cache = output / "audited"
    pending = [row for row in records if row["image_id"] in by_id and cached_row(cache, row, audit_identity) is None]
    if pending:
        auditor = auditor_factory(payload["clip_path"], payload["class_names"])
        size = payload["settings"]["audit_batch_size"]
        for start in tqdm(range(0, len(pending), size), desc="CLIP diagnostic audit"):
            group = pending[start:start + size]
            scores = auditor.score([by_id[row["image_id"]]["final_caption"] for row in group])
            if len(scores) != len(group):
                raise ValueError("CLIP audit batch length mismatch")
            for record, (score, name) in zip(group, scores):
                row = audited_row(by_id[record["image_id"]], score, name,
                    audit_identity, payload["settings"]["semantic_threshold"])
                atomic_json(cache_path(cache, record["image_id"]), {"identity": audit_identity, "row": row})
    rows = available_rows(cache, records, audit_identity)
    complete = len(rows) == len(records)
    destination = output / ("captions.csv" if complete else "captions_audited_partial.csv")
    atomic_csv(destination, rows, COLUMNS)
    status = {"complete": complete, "rows": len(rows), "expected": len(records),
              "generation_identity": generation_identity, "audit_identity": audit_identity,
              "csv_path": str(destination.resolve()), "csv_sha256": file_sha256(destination)}
    atomic_json(output / "status.json", status)
    print(f"Audited rows={len(rows)}/{len(records)}; complete={complete}; CSV={destination}", flush=True)
    return status


def worker_main(request_path, phase):
    import torch
    from transformers import Qwen3VLForConditionalGeneration  # Fail before loading weights on an old runtime.
    payload = json.loads(Path(request_path).read_text(encoding="utf-8"))
    validate_settings(payload["settings"])
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a CUDA GPU for Qwen generation and CLIP audit")
    torch.manual_seed(payload["settings"]["seed"])
    torch.cuda.manual_seed_all(payload["settings"]["seed"])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    output = Path(payload["output"])
    if phase == "generate":
        print("Hashing complete Qwen assets; large checkpoint hashing can take several minutes", flush=True)
        path, params, byte_count = validate_qwen_folder(payload["model_path"], payload["method"])
        protocol = {"schema": 1, "method": payload["method"], "model_id": MODEL_IDS[payload["method"]],
            "model_assets": qwen_asset_identity(path), "stored_parameter_count": params,
            "stored_weight_bytes": byte_count,
            "prompt": CAPTION_PROMPT, "prompt_version": METHOD_VERSION,
            "dtd_split": payload["dtd_split"], "class_names": payload["class_names"],
            "records": [{key: row[key] for key in ("image_id", "split", "image_sha256")} for row in payload["records"]],
            "settings": payload["settings"], "runtime": runtime_versions(), "code_sha256": payload["code_sha256"],
            "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda}
        identity = lock_protocol(output / "generation_protocol.json", protocol)
        free, total = torch.cuda.mem_get_info(0)
        print(f"Stored parameters={params:,}; GPU={total / 1024**3:.1f} GiB; free={free / 1024**3:.1f} GiB", flush=True)
        raw_weight_bytes = params * 2
        missing = any(cached_row(output / "generated", record, identity) is None for record in payload["records"])
        if missing and free < raw_weight_bytes * 1.08:
            raise RuntimeError(f"Insufficient free VRAM even for weights plus a small reserve (~{raw_weight_bytes / 1024**3:.1f} GiB weights). "
                               "32B normally needs a large-memory GPU. No quantization/CPU offload is silently enabled")
        generate_records(payload, identity)
    elif phase == "audit":
        generation = json.loads((output / "generation_protocol.json").read_text(encoding="utf-8"))
        expected_records = [{key: row[key] for key in ("image_id", "split", "image_sha256")} for row in payload["records"]]
        if (generation["method"] != payload["method"] or generation["settings"] != payload["settings"]
                or generation["class_names"] != payload["class_names"] or generation["records"] != expected_records
                or generation["code_sha256"] != payload["code_sha256"]):
            raise ValueError("Audit request does not match the locked generation protocol")
        generation_identity = object_hash(generation)
        protocol = {"schema": 1, "generation_identity": generation_identity,
                    "clip_assets": qwen_asset_identity(payload["clip_path"]),
                    "threshold": payload["settings"]["semantic_threshold"], "policy": "dtd_lexical_v2",
                    "dtype": "float32", "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}
        audit_identity = lock_protocol(output / "audit" / "protocol.json", protocol)
        status = audit_records(payload, generation_identity, audit_identity)
        atomic_json(output / "export_manifest.json", dict(status, model_id=MODEL_IDS[payload["method"]],
            caption_input_policy="dtd_lexical_v2", method=payload["method"], prompt_version=METHOD_VERSION))
    else:
        raise ValueError("Worker phase must be generate or audit")


def worker_environment(runtime_packages=""):
    env = os.environ.copy()
    env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1",
               TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", CUBLAS_WORKSPACE_CONFIG=":4096:8")
    if runtime_packages:
        path = Path(runtime_packages).expanduser().resolve()
        if not (path / "transformers").is_dir():
            raise FileNotFoundError("QWEN_RUNTIME_SITE_PACKAGES must contain transformers/ and compatible dependencies")
        env["PYTHONPATH"] = str(path) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    return env


def check_worker_runtime(env):
    code = (
        "import json, torch, torchvision, transformers, accelerate, tokenizers; "
        "from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, CLIPTextModelWithProjection; "
        "print(json.dumps({'transformers':transformers.__version__, 'torch':torch.__version__, "
        "'cuda':torch.cuda.is_available(), 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"
    )
    result = subprocess.run([sys.executable, "-c", code], env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Qwen3-VL worker import failed. Attach a compatible Transformers >=4.57 runtime and its dependencies; "
            "the old Florence Transformers 4.49 wheels cannot run Qwen3-VL. No packages were installed.\n" + result.stderr[-4000:])
    print(result.stdout.strip())
    info = json.loads(result.stdout.strip().splitlines()[-1])
    if not info["cuda"]:
        raise RuntimeError("Qwen worker imports passed but no CUDA GPU is available. Enable a GPU accelerator")
    return result.stdout.strip()


def run_worker(worker_path, payload_path, phase, env, log_path):
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    with Path(log_path).open("a", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path), phase], env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if code:
        raise RuntimeError(f"{phase} worker exited with code {code}. See {log_path}; committed image caches are preserved")


def summarize_exports(method_outputs, expected_records):
    summaries = []
    exports = {}
    expected = [(row["image_id"], row["split"]) for row in expected_records]
    for method, output in method_outputs.items():
        status_path = Path(output) / "status.json"
        if not status_path.is_file():
            print(f"{method}: not run; no complete CSV yet")
            continue
        status = json.loads(status_path.read_text(encoding="utf-8"))
        if not status["complete"]:
            print(f"{method}: incomplete ({status['rows']}/{status['expected']}); resume before training")
            continue
        path = Path(status["csv_path"])
        if file_sha256(path) != status["csv_sha256"]:
            raise ValueError(f"Export CSV changed after audit: {path}")
        rows = read_csv_rows(path, COLUMNS)
        if [(row["image_id"], row["split"]) for row in rows] != expected:
            raise ValueError(f"Wrong row order/image IDs/split or incomplete CSV: {path}")
        if any(row["generation_method"] != method or row["semantic_audit_status"] != "diagnostic_only"
               or row["caption_input_policy"] != "dtd_lexical_v2" for row in rows):
            raise ValueError(f"Wrong caption export provenance: {path}")
        exports[method] = rows
        for role in ("train", "val", "test"):
            selected = [row for row in rows if row["split"] == role]
            valid = [row for row in selected if bool_value(row["caption_valid"])]
            summaries.append({"method": method, "split": role, "images": len(selected), "valid": len(valid),
                "valid_rate": len(valid) / len(selected), "unique_captions": len({r["final_caption"] for r in valid}),
                "direct_class_hits": sum(bool_value(r["lexical_leakage_detected"]) for r in selected),
                "alias_hits_diagnostic": sum(bool_value(r["alias_hit_diagnostic"]) for r in selected),
                "semantic_flags_diagnostic": sum(bool_value(r["semantic_leakage_detected"]) for r in selected),
                "token_limit_hits": sum(bool_value(r["generation_token_limit_reached"]) for r in selected),
                "csv_sha256": status["csv_sha256"], "csv_path": str(path)})
    return summaries, exports


In [ ]:
validate_settings(SETTINGS)
if type(MAX_NEW_IMAGES) is not int or MAX_NEW_IMAGES < 0:
    raise ValueError("MAX_NEW_IMAGES must be a nonnegative integer")
if not DTD_ROOT.is_dir():
    fallback = Path("/kaggle/input/describable-textures-dataset-dtd/dtd")
    if fallback.is_dir():
        DTD_ROOT = fallback
caption_records, class_names = load_caption_manifest(DTD_ROOT, DTD_SPLIT)
print("Hashing DTD images to lock the same input pixels for both captioners...")
image_fingerprints = fingerprint_records(caption_records)
for record, fingerprint in zip(caption_records, image_fingerprints):
    record["image_sha256"] = fingerprint["sha256"]
for role in ("train", "val", "test"):
    print(role, sum(row["split"] == role for row in caption_records))
print("Total:", len(caption_records), "| classes:", len(class_names))
print("Fixed VLM prompt:", CAPTION_PROMPT)


## Runtime Preflight and Isolated Workers

This preflight checks imports in the SAME Python environment used by the workers, before loading any weights.
If it fails on an old/missing library, attach the required offline runtime. Do not downgrade the kernel to 4.49.
The parent notebook does not import Transformers or keep either Qwen on GPU. No `qwen-vl-utils` package is needed:
PIL pixels are passed directly through the native processor; generated tokens are decoded after trimming the prompt.


In [ ]:
worker_env = worker_environment(QWEN_RUNTIME_SITE_PACKAGES)
check_worker_runtime(worker_env)
clip_dir = resolve_clip_model(CLIP_MODEL_LOCAL_PATH, SEARCH_ROOTS)
print("CLIP diagnostic auditor:", clip_dir)
for method, reference in (("qwen3_vl_8b_instruct", QWEN_8B_MODEL_PATH),
                           ("qwen3_vl_32b_instruct", QWEN_32B_MODEL_PATH)):
    try:
        model_path = resolve_qwen_model(reference, method, SEARCH_ROOTS)
        _, parameters, stored_bytes = validate_qwen_folder(model_path, method)
        print(method, "|", model_path, "| stored parameters:", f"{parameters:,}",
              "| checkpoint GiB:", round(stored_bytes / 1024**3, 1))
    except (OSError, ValueError) as error:
        print(method, "not ready:", error)
print("A missing 32B input does not erase completed 8B outputs; fix its path before running the 32B cell.")


In [ ]:
QWEN_WORKER_SOURCE = '"""Offline, resumable Qwen3-VL caption exports for the existing DTD ablations."""\nfrom __future__ import annotations\n\nimport csv\nfrom collections import defaultdict\nfrom pathlib import PurePosixPath\nimport gc\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport re\nimport struct\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\ndef atomic_csv(path, rows, columns):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\ndef checkpoint_asset_identity(model_dir):\n    model_dir = Path(model_dir)\n    files = sorted(path for path in model_dir.iterdir() if path.is_file() and\n                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))\n    if not any(path.suffix in {".safetensors", ".bin"} for path in files):\n        raise ValueError("Checkpoint has no weight files")\n    return {path.name: file_sha256(path) for path in files}\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef load_caption_manifest(root, split, expected_classes=47, per_class=40):\n    root = Path(root).expanduser().resolve()\n    if not 1 <= int(split) <= 10:\n        raise ValueError("DTD_SPLIT must be in 1..10")\n    images = (root / "images").resolve()\n    names = sorted(path.name for path in images.iterdir() if path.is_dir())\n    if len(names) != expected_classes:\n        raise ValueError(f"Expected {expected_classes} DTD image folders; found {len(names)} at {images}")\n    records, seen = [], set()\n    for role in ("train", "val", "test"):\n        counts = defaultdict(int)\n        for raw in (root / "labels" / f"{role}{split}.txt").read_text(encoding="utf-8-sig").splitlines():\n            text = raw.strip().replace("\\\\", "/")\n            if not text:\n                continue\n            parts = PurePosixPath(text).parts\n            if len(parts) != 2 or parts[0] not in names or any(part in {"..", "."} for part in parts):\n                raise ValueError(f"Unexpected official DTD path: {text!r}")\n            image_id = PurePosixPath(*parts).as_posix()\n            path = images.joinpath(*parts).resolve()\n            if not path.is_relative_to(images) or not path.is_file():\n                raise FileNotFoundError(f"DTD image missing/outside images root: {path}")\n            if image_id in seen:\n                raise ValueError(f"Duplicate image or overlapping official splits: {image_id}")\n            seen.add(image_id)\n            counts[parts[0]] += 1\n            records.append({"image_id": image_id, "image_path": str(path), "split": role})\n        if set(counts) != set(names) or any(count != per_class for count in counts.values()):\n            raise ValueError(f"Official {role}{split}.txt must contain {per_class} images/class")\n    return records, names\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,\n                                    allow_nan=False).encode()).hexdigest()\n\ndef read_csv_rows(path, required=()):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):\n            raise ValueError(f"Missing/duplicate CSV header: {path}")\n        if not set(required).issubset(reader.fieldnames):\n            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")\n        rows = list(reader)\n    if any(None in row or any(value is None for value in row.values()) for row in rows):\n        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")\n    return rows\ndef clean_dtd_caption(text):\n    import re\n    text = str(text).strip()\n    for _ in range(3):\n        before = text\n        text = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|picture|photo(?:graph)?)\\s+"\n                      r"(?:is|shows|depicts|features|contains)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:(?:a|an|the)\\s+)?close[ -]up(?:\\s+view)?\\s+(?:of|showing)\\s+",\n                      "", text, flags=re.I)\n        text = re.sub(r"^(?:a|an|the)(?:\\s+|$)", "", text, flags=re.I)\n        if text == before:\n            break\n    words = re.findall(r"[A-Za-z0-9]+(?:[\'\\u2019][A-Za-z0-9]+)?", text)\n    selected = []\n    normalized_count = 0\n    for word in words:\n        count = len(re.findall(r"[A-Za-z0-9]+", word))\n        if normalized_count + count > 8:\n            break\n        selected.append(word)\n        normalized_count += count\n    return " ".join(selected)\nALIASES = {\n    "banded": ["bands", "banding"], "blotchy": ["blotches", "blotch"], "braided": ["braid", "braids"],\n    "bubbly": ["bubbles", "bubble"], "bumpy": ["bumps", "bump"], "chequered": ["checkered", "checkerboard"],\n    "cobwebbed": ["cobwebs", "cobweb"], "cracked": ["cracks", "crack"],\n    "crosshatched": ["crosshatched", "cross hatching"], "crystalline": ["crystals", "crystal"],\n    "dotted": ["dots", "polka dots"], "fibrous": ["fibers", "fibres"], "flecked": ["flecks"],\n    "freckled": ["freckles"], "frilly": ["frills"], "gauzy": ["gauze"], "gridded": ["grid", "grids"],\n    "grooved": ["grooves"], "honeycombed": ["honeycomb"], "interlaced": ["interlacing"],\n    "knitted": ["knit", "knitting"], "lacelike": ["lace", "lace like"], "lined": ["parallel lines"],\n    "marbled": ["marbling"], "meshed": ["mesh"], "paisley": [], "perforated": ["perforations"],\n    "pitted": ["pits"], "pleated": ["pleats"], "polka-dotted": ["polka dotted", "polka dots"],\n    "porous": ["pores"], "potholed": ["potholes"], "scaly": ["scales"], "smeared": ["smears"],\n    "spiralled": ["spiraled", "spirals", "spiral"], "sprinkled": ["sprinkles"], "stained": ["stains"],\n    "stratified": ["strata"], "striped": ["stripes", "stripe"], "studded": ["studs"],\n    "swirly": ["swirls", "swirl"], "veined": ["veins"], "waffled": ["waffle"],\n    "woven": ["weave", "weaving"], "wrinkled": ["wrinkles"], "zigzagged": ["zigzag", "zig zag"],\n}\n\ndef bool_value(value):\n    return str(value).lower() in {"true", "yes", "1"}\n\ndef load_rgb(path):\n    from PIL import Image, ImageOps\n    with Image.open(path) as source:\n        return ImageOps.exif_transpose(source).convert("RGB")\n\ndef normalize(text):\n    return re.sub(r"[^a-z0-9]+", " ", str(text).replace("_", " ").lower()).strip()\n\ndef term_hits(text, names, aliases=False):\n    padded = " " + normalize(text) + " "\n    terms = set(normalize(name) for name in names)\n    if aliases:\n        terms.update(normalize(term) for name in names for term in ALIASES.get(name, []))\n    return sorted(term for term in terms if term and " " + term + " " in padded)\n\ndef valid_short(text, names):\n    return 5 <= len(normalize(text).split()) <= 8 and not term_hits(text, names)\n\n\nMETHOD_VERSION = "qwen3vl_dtd_short_caption_v1"\nMODEL_IDS = {\n    "qwen3_vl_8b_instruct": "Qwen/Qwen3-VL-8B-Instruct",\n    "qwen3_vl_32b_instruct": "Qwen/Qwen3-VL-32B-Instruct",\n}\nCAPTION_PROMPT = (\n    "Describe only the visible surface texture in 5 to 8 English words. "\n    "Focus on visible colors, local shapes, spacing, repetition, and surface structure. "\n    "Do not identify the texture category or guess the object, use, location, or hidden material. "\n    "Start directly with descriptive words, not an article or an image introduction. "\n    "Output only one short description, without a heading, explanation, or quotation marks."\n)\nCOLUMNS = [\n    "image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",\n    "lexical_leakage_detected", "semantic_leakage_detected", "semantic_leakage_score",\n    "semantic_leakage_class", "retry_count", "vlm_model", "prompt_version",\n    "caption_input_policy", "semantic_audit_status", "captioner_fingerprint", "generation_method",\n    "source_long_caption", "source_word_count", "word_count", "cleanup_changed_text",\n    "alias_hit_diagnostic", "generation_token_limit_reached", "effective_batch_size", "audit_fingerprint",\n]\n\n\ndef local_model_configs(roots):\n    seen = set()\n    for root in roots:\n        root = Path(root).expanduser()\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {\n                "images", ".git", "__pycache__", "node_modules", ".cache"})\n            if len(Path(current).relative_to(root).parts) >= 9:\n                dirs[:] = []\n            if "config.json" not in files:\n                continue\n            path = Path(current).resolve()\n            if str(path) not in seen:\n                seen.add(str(path))\n                yield path\n\n\ndef weight_files(path):\n    path = Path(path).resolve()\n    index = path / "model.safetensors.index.json"\n    if index.is_file():\n        names = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())\n        if not names:\n            raise ValueError(f"Empty safetensors weight index: {index}")\n        if any(not isinstance(name, str) or "/" in name or "\\\\" in name or Path(name).is_absolute() for name in names):\n            raise ValueError(f"Weight index must reference shard filenames inside its model folder: {index}")\n        files = [path / name for name in sorted(names)]\n    else:\n        files = [path / "model.safetensors"]\n    for file in files:\n        # HF snapshot files may be read-only symlinks into a sibling blobs directory.\n        if not file.is_file() or file.suffix != ".safetensors":\n            raise FileNotFoundError(f"Missing/unsafe local safetensors shard: {file}")\n    return files\n\n\ndef safetensors_parameter_count(files):\n    # Read headers only, not tensor contents; verify the attached model\'s actual scale.\n    count = 0\n    for path in files:\n        with Path(path).open("rb") as handle:\n            raw_length = handle.read(8)\n            if len(raw_length) != 8:\n                raise ValueError(f"Truncated safetensors header: {path}")\n            length = struct.unpack("<Q", raw_length)[0]\n            if not 2 <= length <= 64 * 1024 * 1024 or length + 8 > Path(path).stat().st_size:\n                raise ValueError(f"Invalid safetensors header length: {path}")\n            header = json.loads(handle.read(length))\n        for name, tensor in header.items():\n            if name == "__metadata__":\n                continue\n            if tensor.get("dtype") not in {"F16", "BF16", "F32"}:\n                raise ValueError("Use original floating-point weights, not quantized/FP8 weights, for this comparison")\n            shape = tensor.get("shape", [])\n            if not all(type(size) is int and size >= 0 for size in shape):\n                raise ValueError(f"Invalid tensor shape: {path}: {name}")\n            count += math.prod(shape)\n    return count\n\n\ndef validate_qwen_folder(path, method):\n    path = Path(path).expanduser().resolve()\n    if method not in MODEL_IDS:\n        raise ValueError(f"Unknown Qwen method: {method}")\n    config = json.loads((path / "config.json").read_text(encoding="utf-8"))\n    if config.get("model_type") != "qwen3_vl" or config.get("quantization_config"):\n        raise ValueError("Expected original dense Qwen3-VL, not Qwen2.5, pure-text Qwen, MoE or a quantized model")\n    if "thinking" in str(path).lower():\n        raise ValueError("Attach the Instruct checkpoint, not Thinking")\n    for name in ("tokenizer_config.json", "preprocessor_config.json"):\n        if not (path / name).is_file():\n            raise FileNotFoundError(f"Missing local processor/tokenizer asset: {path / name}")\n    if not (path / "tokenizer.json").is_file():\n        raise FileNotFoundError(f"Missing tokenizer.json in {path}")\n    tokenizer = json.loads((path / "tokenizer_config.json").read_text(encoding="utf-8"))\n    template_json = path / "chat_template.json"\n    json_template = json.loads(template_json.read_text(encoding="utf-8")).get("chat_template") if template_json.is_file() else None\n    if not tokenizer.get("chat_template") and not (path / "chat_template.jinja").is_file() and not json_template:\n        raise FileNotFoundError(f"Missing Qwen chat template in {path}")\n    files = weight_files(path)\n    count = safetensors_parameter_count(files)\n    low, high = (7_000_000_000, 11_000_000_000) if "8b" in method else (28_000_000_000, 38_000_000_000)\n    if not low <= count <= high:\n        raise ValueError(f"Wrong checkpoint size for {MODEL_IDS[method]}: {count:,} stored parameters")\n    return path, count, sum(file.stat().st_size for file in files)\n\n\ndef resolve_qwen_model(reference, method, roots):\n    if str(reference).strip():\n        path, _, _ = validate_qwen_folder(reference, method)\n        return path\n    needle = MODEL_IDS[method].split("/")[-1].lower()\n    matches = [path for path in local_model_configs(roots) if needle in str(path).lower()]\n    if len(matches) != 1:\n        raise FileNotFoundError(\n            f"Set the exact local path for {MODEL_IDS[method]}. Found {len(matches)} named matches. "\n            "Attach its complete Hugging Face folder as Kaggle Input; Florence weights are not Qwen weights. "\n            "No download is attempted.")\n    return validate_qwen_folder(matches[0], method)[0]\n\n\ndef resolve_clip_model(reference, roots):\n    def is_clip(path):\n        try:\n            return json.loads((path / "config.json").read_text(encoding="utf-8")).get("model_type") == "clip"\n        except (OSError, ValueError):\n            return False\n    if str(reference).strip() and Path(reference).is_dir():\n        path = Path(reference).resolve()\n        if not is_clip(path):\n            raise ValueError(f"Not a full local CLIP checkpoint: {path}")\n        return path\n    matches = [path for path in local_model_configs(roots) if is_clip(path)]\n    if len(matches) != 1:\n        raise FileNotFoundError("Set CLIP_MODEL_LOCAL_PATH to the exact full CLIP ViT-B/32 folder")\n    return matches[0]\n\n\ndef qwen_asset_identity(path):\n    result = checkpoint_asset_identity(path)\n    for template in sorted(Path(path).glob("*.jinja")):\n        result[template.name] = file_sha256(template)\n    return result\n\n\ndef fingerprint_records(records):\n    return [{"image_id": row["image_id"], "split": row["split"],\n             "sha256": file_sha256(row["image_path"])} for row in records]\n\n\ndef runtime_versions():\n    result = {}\n    for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers", "Pillow", "numpy"):\n        result[name] = importlib.metadata.version(name)\n    return result\n\n\ndef validate_settings(settings):\n    for key in ("batch_size", "max_new_tokens", "min_pixels", "max_pixels", "audit_batch_size"):\n        if type(settings[key]) is not int or settings[key] < 1:\n            raise ValueError(f"{key} must be a positive integer")\n    if settings["min_pixels"] > settings["max_pixels"]:\n        raise ValueError("min_pixels must not exceed max_pixels")\n    if type(settings["seed"]) is not int or not 0 <= settings["seed"] < 2**32:\n        raise ValueError("seed must be an integer in [0, 2**32)")\n    if settings["dtype"] not in {"bfloat16", "float16"}:\n        raise ValueError("Use the same bfloat16 or float16 dtype for both captioners")\n    if not math.isfinite(settings["semantic_threshold"]) or not -1 <= settings["semantic_threshold"] <= 1:\n        raise ValueError("semantic_threshold must be finite and in [-1, 1]")\n\n\ndef lock_protocol(path, protocol):\n    path = Path(path)\n    if path.is_file():\n        if json.loads(path.read_text(encoding="utf-8")) != protocol:\n            raise ValueError(f"Changed images/model/prompt/runtime/settings. Use a new output directory: {path.parent}")\n    else:\n        if path.parent.exists() and any(path.parent.iterdir()):\n            raise ValueError(f"Refusing to adopt cached output without its protocol: {path.parent}")\n        atomic_json(path, protocol)\n    return object_hash(protocol)\n\n\ndef cache_path(directory, image_id):\n    return Path(directory) / (object_hash(image_id) + ".json")\n\n\ndef cached_row(directory, record, identity):\n    path = cache_path(directory, record["image_id"])\n    if not path.is_file():\n        return None\n    result = json.loads(path.read_text(encoding="utf-8"))\n    if (result.get("identity") != identity or result.get("row", {}).get("image_id") != record["image_id"]\n            or result["row"].get("split") != record["split"]):\n        raise ValueError(f"Wrong cached image/protocol: {path}")\n    return result["row"]\n\n\ndef generation_row(record, response, method, identity, names, token_limit=False, batch_size=1):\n    text = clean_dtd_caption(response)\n    lexical = bool(term_hits(text, names))\n    return {\n        "image_id": record["image_id"], "split": record["split"], "raw_caption": text, "final_caption": text,\n        "caption_valid": valid_short(text, names), "leakage_detected": lexical,\n        "lexical_leakage_detected": lexical, "semantic_leakage_detected": False,\n        "semantic_leakage_score": "", "semantic_leakage_class": "", "retry_count": 0,\n        "vlm_model": MODEL_IDS[method], "prompt_version": METHOD_VERSION,\n        "caption_input_policy": "dtd_lexical_v2", "semantic_audit_status": "pending",\n        "captioner_fingerprint": identity, "generation_method": method,\n        "source_long_caption": str(response), "source_word_count": len(normalize(response).split()),\n        "word_count": len(normalize(text).split()), "cleanup_changed_text": str(response).strip() != text,\n        "alias_hit_diagnostic": bool(term_hits(text, names, aliases=True)),\n        "generation_token_limit_reached": bool(token_limit), "effective_batch_size": batch_size,\n        "audit_fingerprint": "",\n    }\n\n\ndef audited_row(row, score, class_name, identity, threshold):\n    if not math.isfinite(score) or not -1.00001 <= score <= 1.00001 or not class_name:\n        raise ValueError("Invalid CLIP class-audit result; do not fill missing scores with zero")\n    result = dict(row)\n    result.update(semantic_leakage_score=round(score, 6), semantic_leakage_class=class_name,\n                  semantic_leakage_detected=score >= threshold,\n                  semantic_audit_status="diagnostic_only", audit_fingerprint=identity)\n    # dtd_lexical_v2 deliberately does not hard-reject text using class-CLIP similarity.\n    return result\n\n\ndef caption_messages(images, prompt=CAPTION_PROMPT):\n    return [{"role": "user", "content": [{"type": "image", "image": image},\n            {"type": "text", "text": prompt}]} for image in images]\n\n\nclass FrozenQwenCaptioner:\n    def __init__(self, path, settings):\n        import torch\n        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration\n        self.device = torch.device("cuda:0")\n        self.settings = settings\n        dtype = getattr(torch, settings["dtype"])\n        if dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():\n            raise RuntimeError("This GPU lacks BF16 support. Set dtype=float16 for BOTH models in a new output directory")\n        self.processor = AutoProcessor.from_pretrained(str(path), local_files_only=True, trust_remote_code=False,\n            min_pixels=settings["min_pixels"], max_pixels=settings["max_pixels"], use_fast=True)\n        self.processor.tokenizer.padding_side = "left"\n        self.model, info = Qwen3VLForConditionalGeneration.from_pretrained(str(path), local_files_only=True,\n            trust_remote_code=False, dtype=dtype, device_map={"": str(self.device)},\n            attn_implementation="sdpa", low_cpu_mem_usage=True, output_loading_info=True)\n        if info.get("missing_keys") or info.get("mismatched_keys"):\n            raise ValueError(f"Incomplete Qwen learned weights: {info}")\n        self.model.eval().requires_grad_(False)\n\n    def generate(self, images):\n        import torch\n        conversations = [[message] for message in caption_messages(images)]\n        prompts = self.processor.apply_chat_template(conversations, tokenize=False, add_generation_prompt=True)\n        inputs = self.processor(text=prompts, images=images, padding=True, return_tensors="pt",\n            min_pixels=self.settings["min_pixels"], max_pixels=self.settings["max_pixels"])\n        inputs.pop("token_type_ids", None)\n        inputs = {key: value.to(self.device) if torch.is_tensor(value) else value for key, value in inputs.items()}\n        with torch.inference_mode():\n            output = self.model.generate(**inputs, max_new_tokens=self.settings["max_new_tokens"],\n                do_sample=False, num_beams=1, use_cache=True, temperature=None, top_p=None, top_k=None)\n        suffix = output[:, inputs["input_ids"].shape[1]:]\n        texts = self.processor.batch_decode(suffix, skip_special_tokens=True, clean_up_tokenization_spaces=False)\n        eos = self.model.generation_config.eos_token_id\n        eos = set(eos if isinstance(eos, (list, tuple)) else [eos])\n        limits = [len(tokens) >= self.settings["max_new_tokens"] and not any(token in eos for token in tokens)\n                  for tokens in suffix.cpu().tolist()]\n        if len(texts) != len(images):\n            raise ValueError("Qwen output count differs from input image count")\n        return list(zip(texts, limits))\n\n\nclass ClipClassAuditor:\n    def __init__(self, path, names):\n        import torch\n        from transformers import AutoTokenizer, CLIPTextModelWithProjection\n        self.tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)\n        self.model, info = CLIPTextModelWithProjection.from_pretrained(str(path), local_files_only=True,\n            output_loading_info=True)\n        missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n        if missing or info.get("mismatched_keys"):\n            raise ValueError(f"Incomplete CLIP text tower/projection: {missing}")\n        self.model = self.model.to("cuda:0").eval().requires_grad_(False)\n        self.names = names\n        prompts = [template.format(name=name.replace("_", " ").replace("-", " ")) for name in names\n                   for template in ("a photo of {name}", "a scene showing {name}")]\n        self.class_vectors = torch.nn.functional.normalize(self.encode(prompts).reshape(len(names), 2, -1).mean(1), dim=1)\n\n    def encode(self, texts):\n        import torch\n        tokens = self.tokenizer(list(texts), padding=True, truncation=True,\n            max_length=self.model.config.max_position_embeddings, return_tensors="pt")\n        tokens = {key: value.to("cuda:0") for key, value in tokens.items()}\n        with torch.inference_mode():\n            features = self.model(**tokens).text_embeds\n        return torch.nn.functional.normalize(features.float(), dim=1)\n\n    def score(self, texts):\n        scores, indices = (self.encode(texts) @ self.class_vectors.T).max(dim=1)\n        return [(float(score), self.names[index]) for score, index in zip(scores.cpu(), indices.cpu().tolist())]\n\n\ndef available_rows(directory, records, identity):\n    return [row for record in records if (row := cached_row(directory, record, identity)) is not None]\n\n\ndef generate_records(payload, identity, captioner_factory=FrozenQwenCaptioner):\n    import torch\n    from tqdm.auto import tqdm\n    output = Path(payload["output"])\n    cache = output / "generated"\n    records, settings = payload["records"], payload["settings"]\n    todo = [row for row in records if cached_row(cache, row, identity) is None]\n    cached_count = len(records) - len(todo)\n    limit = payload.get("max_new_images", 0)\n    if limit:\n        todo = todo[:limit]\n    started = time.perf_counter()\n    print(f"{payload[\'method\']}: cached={cached_count}, queued={len(todo)}", flush=True)\n    if todo:\n        model = captioner_factory(payload["model_path"], settings)\n        current, offset = settings["batch_size"], 0\n        progress = tqdm(total=len(todo), desc=payload["method"], unit="image")\n        try:\n            while offset < len(todo):\n                group = todo[offset:offset + current]\n                images = []\n                try:\n                    for row in group:\n                        if file_sha256(row["image_path"]) != row["image_sha256"]:\n                            raise ValueError(f"Image changed after manifest hashing: {row[\'image_id\']}")\n                        images.append(load_rgb(row["image_path"]))\n                    responses = model.generate(images)\n                    if len(responses) != len(group):\n                        raise ValueError("Incorrect caption batch length; batch not committed")\n                except torch.cuda.OutOfMemoryError:\n                    if current == 1:\n                        raise RuntimeError("Qwen OOM at batch 1. Free GPU memory or use a larger GPU; completed images are cached") from None\n                    current = max(1, current // 2)\n                    gc.collect()\n                    torch.cuda.empty_cache()\n                    print(f"CUDA OOM: generation batch reduced to {current}; no image skipped", flush=True)\n                    continue\n                finally:\n                    for image in images:\n                        image.close()\n                for record, (text, token_limit) in zip(group, responses):\n                    row = generation_row(record, text, payload["method"], identity,\n                        payload["class_names"], token_limit, len(group))\n                    atomic_json(cache_path(cache, record["image_id"]), {"identity": identity, "row": row})\n                offset += len(group)\n                progress.update(len(group))\n        finally:\n            progress.close()\n            del model\n            gc.collect()\n            torch.cuda.empty_cache()\n    rows = available_rows(cache, records, identity)\n    atomic_csv(output / "captions_raw_partial.csv", rows, COLUMNS)\n    atomic_json(output / "generation_status.json", {"identity": identity, "rows": len(rows),\n                "expected": len(records), "complete": len(rows) == len(records),\n                "new_images_this_invocation": len(todo), "elapsed_seconds_this_invocation": time.perf_counter() - started})\n\n\ndef audit_records(payload, generation_identity, audit_identity, auditor_factory=ClipClassAuditor):\n    from tqdm.auto import tqdm\n    output, records = Path(payload["output"]), payload["records"]\n    raw_rows = available_rows(output / "generated", records, generation_identity)\n    by_id = {row["image_id"]: row for row in raw_rows}\n    cache = output / "audited"\n    pending = [row for row in records if row["image_id"] in by_id and cached_row(cache, row, audit_identity) is None]\n    if pending:\n        auditor = auditor_factory(payload["clip_path"], payload["class_names"])\n        size = payload["settings"]["audit_batch_size"]\n        for start in tqdm(range(0, len(pending), size), desc="CLIP diagnostic audit"):\n            group = pending[start:start + size]\n            scores = auditor.score([by_id[row["image_id"]]["final_caption"] for row in group])\n            if len(scores) != len(group):\n                raise ValueError("CLIP audit batch length mismatch")\n            for record, (score, name) in zip(group, scores):\n                row = audited_row(by_id[record["image_id"]], score, name,\n                    audit_identity, payload["settings"]["semantic_threshold"])\n                atomic_json(cache_path(cache, record["image_id"]), {"identity": audit_identity, "row": row})\n    rows = available_rows(cache, records, audit_identity)\n    complete = len(rows) == len(records)\n    destination = output / ("captions.csv" if complete else "captions_audited_partial.csv")\n    atomic_csv(destination, rows, COLUMNS)\n    status = {"complete": complete, "rows": len(rows), "expected": len(records),\n              "generation_identity": generation_identity, "audit_identity": audit_identity,\n              "csv_path": str(destination.resolve()), "csv_sha256": file_sha256(destination)}\n    atomic_json(output / "status.json", status)\n    print(f"Audited rows={len(rows)}/{len(records)}; complete={complete}; CSV={destination}", flush=True)\n    return status\n\n\ndef worker_main(request_path, phase):\n    import torch\n    from transformers import Qwen3VLForConditionalGeneration  # Fail before loading weights on an old runtime.\n    payload = json.loads(Path(request_path).read_text(encoding="utf-8"))\n    validate_settings(payload["settings"])\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a CUDA GPU for Qwen generation and CLIP audit")\n    torch.manual_seed(payload["settings"]["seed"])\n    torch.cuda.manual_seed_all(payload["settings"]["seed"])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    output = Path(payload["output"])\n    if phase == "generate":\n        print("Hashing complete Qwen assets; large checkpoint hashing can take several minutes", flush=True)\n        path, params, byte_count = validate_qwen_folder(payload["model_path"], payload["method"])\n        protocol = {"schema": 1, "method": payload["method"], "model_id": MODEL_IDS[payload["method"]],\n            "model_assets": qwen_asset_identity(path), "stored_parameter_count": params,\n            "stored_weight_bytes": byte_count,\n            "prompt": CAPTION_PROMPT, "prompt_version": METHOD_VERSION,\n            "dtd_split": payload["dtd_split"], "class_names": payload["class_names"],\n            "records": [{key: row[key] for key in ("image_id", "split", "image_sha256")} for row in payload["records"]],\n            "settings": payload["settings"], "runtime": runtime_versions(), "code_sha256": payload["code_sha256"],\n            "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda}\n        identity = lock_protocol(output / "generation_protocol.json", protocol)\n        free, total = torch.cuda.mem_get_info(0)\n        print(f"Stored parameters={params:,}; GPU={total / 1024**3:.1f} GiB; free={free / 1024**3:.1f} GiB", flush=True)\n        raw_weight_bytes = params * 2\n        missing = any(cached_row(output / "generated", record, identity) is None for record in payload["records"])\n        if missing and free < raw_weight_bytes * 1.08:\n            raise RuntimeError(f"Insufficient free VRAM even for weights plus a small reserve (~{raw_weight_bytes / 1024**3:.1f} GiB weights). "\n                               "32B normally needs a large-memory GPU. No quantization/CPU offload is silently enabled")\n        generate_records(payload, identity)\n    elif phase == "audit":\n        generation = json.loads((output / "generation_protocol.json").read_text(encoding="utf-8"))\n        expected_records = [{key: row[key] for key in ("image_id", "split", "image_sha256")} for row in payload["records"]]\n        if (generation["method"] != payload["method"] or generation["settings"] != payload["settings"]\n                or generation["class_names"] != payload["class_names"] or generation["records"] != expected_records\n                or generation["code_sha256"] != payload["code_sha256"]):\n            raise ValueError("Audit request does not match the locked generation protocol")\n        generation_identity = object_hash(generation)\n        protocol = {"schema": 1, "generation_identity": generation_identity,\n                    "clip_assets": qwen_asset_identity(payload["clip_path"]),\n                    "threshold": payload["settings"]["semantic_threshold"], "policy": "dtd_lexical_v2",\n                    "dtype": "float32", "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}\n        audit_identity = lock_protocol(output / "audit" / "protocol.json", protocol)\n        status = audit_records(payload, generation_identity, audit_identity)\n        atomic_json(output / "export_manifest.json", dict(status, model_id=MODEL_IDS[payload["method"]],\n            caption_input_policy="dtd_lexical_v2", method=payload["method"], prompt_version=METHOD_VERSION))\n    else:\n        raise ValueError("Worker phase must be generate or audit")\n\n\ndef worker_environment(runtime_packages=""):\n    env = os.environ.copy()\n    env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1",\n               TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", CUBLAS_WORKSPACE_CONFIG=":4096:8")\n    if runtime_packages:\n        path = Path(runtime_packages).expanduser().resolve()\n        if not (path / "transformers").is_dir():\n            raise FileNotFoundError("QWEN_RUNTIME_SITE_PACKAGES must contain transformers/ and compatible dependencies")\n        env["PYTHONPATH"] = str(path) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")\n    return env\n\n\ndef check_worker_runtime(env):\n    code = (\n        "import json, torch, torchvision, transformers, accelerate, tokenizers; "\n        "from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, CLIPTextModelWithProjection; "\n        "print(json.dumps({\'transformers\':transformers.__version__, \'torch\':torch.__version__, "\n        "\'cuda\':torch.cuda.is_available(), \'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"\n    )\n    result = subprocess.run([sys.executable, "-c", code], env=env, capture_output=True, text=True)\n    if result.returncode:\n        raise RuntimeError("Qwen3-VL worker import failed. Attach a compatible Transformers >=4.57 runtime and its dependencies; "\n            "the old Florence Transformers 4.49 wheels cannot run Qwen3-VL. No packages were installed.\\n" + result.stderr[-4000:])\n    print(result.stdout.strip())\n    info = json.loads(result.stdout.strip().splitlines()[-1])\n    if not info["cuda"]:\n        raise RuntimeError("Qwen worker imports passed but no CUDA GPU is available. Enable a GPU accelerator")\n    return result.stdout.strip()\n\n\ndef run_worker(worker_path, payload_path, phase, env, log_path):\n    Path(log_path).parent.mkdir(parents=True, exist_ok=True)\n    with Path(log_path).open("a", encoding="utf-8") as log:\n        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path), phase], env=env,\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)\n        try:\n            for line in process.stdout:\n                print(line, end="", flush=True)\n                log.write(line)\n                log.flush()\n            code = process.wait()\n        except BaseException:\n            process.terminate()\n            process.wait()\n            raise\n        finally:\n            process.stdout.close()\n    if code:\n        raise RuntimeError(f"{phase} worker exited with code {code}. See {log_path}; committed image caches are preserved")\n\n\ndef summarize_exports(method_outputs, expected_records):\n    summaries = []\n    exports = {}\n    expected = [(row["image_id"], row["split"]) for row in expected_records]\n    for method, output in method_outputs.items():\n        status_path = Path(output) / "status.json"\n        if not status_path.is_file():\n            print(f"{method}: not run; no complete CSV yet")\n            continue\n        status = json.loads(status_path.read_text(encoding="utf-8"))\n        if not status["complete"]:\n            print(f"{method}: incomplete ({status[\'rows\']}/{status[\'expected\']}); resume before training")\n            continue\n        path = Path(status["csv_path"])\n        if file_sha256(path) != status["csv_sha256"]:\n            raise ValueError(f"Export CSV changed after audit: {path}")\n        rows = read_csv_rows(path, COLUMNS)\n        if [(row["image_id"], row["split"]) for row in rows] != expected:\n            raise ValueError(f"Wrong row order/image IDs/split or incomplete CSV: {path}")\n        if any(row["generation_method"] != method or row["semantic_audit_status"] != "diagnostic_only"\n               or row["caption_input_policy"] != "dtd_lexical_v2" for row in rows):\n            raise ValueError(f"Wrong caption export provenance: {path}")\n        exports[method] = rows\n        for role in ("train", "val", "test"):\n            selected = [row for row in rows if row["split"] == role]\n            valid = [row for row in selected if bool_value(row["caption_valid"])]\n            summaries.append({"method": method, "split": role, "images": len(selected), "valid": len(valid),\n                "valid_rate": len(valid) / len(selected), "unique_captions": len({r["final_caption"] for r in valid}),\n                "direct_class_hits": sum(bool_value(r["lexical_leakage_detected"]) for r in selected),\n                "alias_hits_diagnostic": sum(bool_value(r["alias_hit_diagnostic"]) for r in selected),\n                "semantic_flags_diagnostic": sum(bool_value(r["semantic_leakage_detected"]) for r in selected),\n                "token_limit_hits": sum(bool_value(r["generation_token_limit_reached"]) for r in selected),\n                "csv_sha256": status["csv_sha256"], "csv_path": str(path)})\n    return summaries, exports\n\n\nif __name__ == "__main__":\n    worker_main(sys.argv[1], sys.argv[2])\n'


def run_caption_method(method, reference):
    model_path = resolve_qwen_model(reference, method, SEARCH_ROOTS)
    runtime_dir = OUTPUT_ROOT / "runtime"
    runtime_dir.mkdir(parents=True, exist_ok=True)
    worker_path = runtime_dir / "qwen_dtd_worker.py"
    worker_path.write_text(QWEN_WORKER_SOURCE, encoding="utf-8")
    request_path = runtime_dir / f"{method}.request.json"
    payload = {"method": method, "model_path": str(model_path), "clip_path": str(clip_dir),
               "output": str(METHOD_OUTPUTS[method].resolve()), "records": caption_records,
               "class_names": class_names, "dtd_split": DTD_SPLIT, "settings": SETTINGS,
               "max_new_images": MAX_NEW_IMAGES, "code_sha256": object_hash(QWEN_WORKER_SOURCE)}
    atomic_json(request_path, payload)
    log_path = runtime_dir / f"{method}.log"
    # Audit starts only after the Qwen process exits, so its GPU weights are released.
    run_worker(worker_path, request_path, "generate", worker_env, log_path)
    run_worker(worker_path, request_path, "audit", worker_env, log_path)
    return json.loads((METHOD_OUTPUTS[method] / "status.json").read_text(encoding="utf-8"))


## 1. Qwen3-VL-8B-Instruct

Run/resume 8B first. Cached images are not generated again, including flagged captions: no hidden retries
or generic replacements turn a failure into fabricated evidence. If an inference error is not CUDA OOM,
the worker stops visibly. OOM halves batches; OOM at batch 1 stops and preserves committed caches.


In [ ]:
if RUN_8B:
    status_8b = run_caption_method("qwen3_vl_8b_instruct", QWEN_8B_MODEL_PATH)
    print(status_8b)
else:
    print("8B generation disabled by RUN_8B=False")


## 2. Qwen3-VL-32B-Instruct

Run/resume 32B with the SAME prompt/settings. The previous model is not resident on GPU.
This cell requires a real 32B checkpoint; it never silently substitutes 8B or Florence.
If only a smoke test was run, reset `MAX_NEW_IMAGES=0` and rerun these two generation cells to finish.


In [ ]:
if RUN_32B:
    status_32b = run_caption_method("qwen3_vl_32b_instruct", QWEN_32B_MODEL_PATH)
    print(status_32b)
else:
    print("32B generation disabled by RUN_32B=False")


## Export Contract and Policy

Exports retain the required ablation columns and `caption_input_policy=dtd_lexical_v2`.
`raw_caption` and `final_caption` hold the SAME cleaned short model input, because the existing classifier
cleans raw_caption before encoding it. The full untouched Qwen response is retained as `source_long_caption`.
Generic openings are removed and at most 8 normalized words are kept using the EXISTING cleaner.
Captions with fewer than 5 words or a canonical DTD class term are masked, not invented or repaired.
Original response length, cleanup, alias hits and token-limit stops remain visible in the audit columns.

CLIP compares the final short text with the same two class templates as earlier experiments. Its similarity/flag
is diagnostic ONLY and does not alter caption_valid or leakage_detected. Alias hits are also diagnostic-only,
so this is the same outer policy, not a newly stricter hidden mask. No synonym/semantic no-leak guarantee is made.
These are **generated captions**, not human-verified ground truth. Coverage and CLIP scores are not classification metrics.


In [ ]:
import pandas as pd
from IPython.display import display

summary_rows, complete_exports = summarize_exports(METHOD_OUTPUTS, caption_records)
export_dir = OUTPUT_ROOT / "exports"
published = {}
for method, rows in complete_exports.items():
    filename = "captions_qwen3vl_8b_instruct.csv" if "8b" in method else "captions_qwen3vl_32b_instruct.csv"
    destination = export_dir / filename
    atomic_csv(destination, rows, COLUMNS)
    published[method] = {"csv_path": str(destination.resolve()), "csv_sha256": file_sha256(destination),
                         "rows": len(rows), "captioner_fingerprint": rows[0]["captioner_fingerprint"],
                         "audit_fingerprint": rows[0]["audit_fingerprint"]}
    print("COMPLETE EXPORT:", destination, "| rows:", len(rows))
if summary_rows:
    summary = pd.DataFrame(summary_rows)
    display(summary.drop(columns=["csv_sha256", "csv_path"]))
    atomic_csv(OUTPUT_ROOT / "caption_generation_summary.csv", summary_rows, list(summary_rows[0]))
atomic_json(OUTPUT_ROOT / "caption_methods_manifest.json", {
    "schema": 1, "dataset": "DTD", "dtd_split": DTD_SPLIT, "prompt": CAPTION_PROMPT,
    "caption_input_policy": "dtd_lexical_v2", "methods": published,
    "both_complete": len(published) == 2, "captioner_finetuning": False, "classifier_training": False})
if len(published) != 2:
    print("Both methods are not complete yet. Resume the missing model; never train from partial CSVs.")


In [ ]:
def review_train_image(index=0):
    training = [row for row in caption_records if row["split"] == "train"]
    if not 0 <= index < len(training):
        raise IndexError("Choose an official training-image index")
    record = training[index]
    image = load_rgb(record["image_path"])
    image.thumbnail((700, 500))
    display(image)
    for method, rows in complete_exports.items():
        row = next(row for row in rows if row["image_id"] == record["image_id"])
        print(method)
        print("  Response:", row["source_long_caption"])
        print("  Model input:", row["final_caption"], "| valid:", row["caption_valid"],
              "| alias diagnostic:", row["alias_hit_diagnostic"])
    image.close()

review_train_image(0)  # Review training images, not test examples for prompt tuning.


## Paired Caption Diagnostics

This comparison checks the same image IDs/order/splits and reports natural availability masks.
It does NOT hold the selected context fixed: full_both uses caption similarity in selection, so changing CSVs
can change text fusion, selected context images and the training trajectory. Any downstream result is a
whole-pipeline captioner ablation. These diagnostics do not measure visual correctness or prove label leakage.


In [ ]:
if len(complete_exports) == 2:
    a = complete_exports["qwen3_vl_8b_instruct"]
    b = complete_exports["qwen3_vl_32b_instruct"]
    paired = [{"image_id": left["image_id"], "split": left["split"],
               "caption_8b": left["final_caption"], "caption_32b": right["final_caption"],
               "valid_8b": bool_value(left["caption_valid"]), "valid_32b": bool_value(right["caption_valid"]),
               "same_text": normalize(left["final_caption"]) == normalize(right["final_caption"])}
              for left, right in zip(a, b)]
    atomic_csv(OUTPUT_ROOT / "paired_caption_audit.csv", paired, list(paired[0]))
    paired_frame = pd.DataFrame(paired)
    display(paired_frame.groupby("split")[["valid_8b", "valid_32b", "same_text"]].mean())
    print("No test accuracy or F1 has been measured by this notebook.")
else:
    print("Finish both methods before paired caption diagnostics.")


## Train Two Fresh M-TACS Ablations

Use the ORIGINAL **notebook 05 full_both**, not the four Florence-method notebooks with fixed filename/SHA guards.
Make two copies of notebook 05. Attach the two complete exports to Kaggle, one chosen CSV per classifier run.
At the END of each copy's configuration cell, after its default caption variables, set:

```python
# Copy A: Qwen3-VL-8B captions
CAPTIONS_INPUT_PATH = "/kaggle/input/YOUR_ATTACHED_DATASET/captions_qwen3vl_8b_instruct.csv"
CAPTION_INPUT_POLICY = "dtd_lexical_v2"
GENERATE_CAPTIONS_IF_MISSING = False
os.environ["DTD_FULL_BOTH_OUTPUT"] = "/kaggle/working/outputs/dtd_full_both_qwen3vl_8b_seed42"
```

```python
# Copy B: Qwen3-VL-32B captions
CAPTIONS_INPUT_PATH = "/kaggle/input/YOUR_ATTACHED_DATASET/captions_qwen3vl_32b_instruct.csv"
CAPTION_INPUT_POLICY = "dtd_lexical_v2"
GENERATE_CAPTIONS_IF_MISSING = False
os.environ["DTD_FULL_BOTH_OUTPUT"] = "/kaggle/working/outputs/dtd_full_both_qwen3vl_32b_seed42"
```

Set the output variable BEFORE running notebook 05's dataset/output-directory cell. Keep DTD split 1,
pool seed/training seed 42, K=1, 50 epochs, matched batch/optimizer/loss/augmentation and validation-loss selection.
Use a new output directory and fresh classifier/text caches for each CSV; never resume Florence/8B weights as a 32B run.
Neither Qwen checkpoint is needed during classifier training when captions are already precomputed.
The shared raw CSV is read-only; `shared/caption_model_inputs.csv` records actual cleaned text/masks.

First check training-only coverage and visible correctness, then run the locked comparisons. Do not adjust
prompts, thresholds or decoding to chase repeated test F1. Consider multiple seeds/splits before claiming a stable gain.
A bigger captioner, longer response, higher lexical validity or higher CLIP class score does not guarantee better M-TACS.

Official references: [Qwen3-VL repository](https://github.com/QwenLM/Qwen3-VL),
[8B model](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct),
[32B model](https://huggingface.co/Qwen/Qwen3-VL-32B-Instruct),
[native Transformers processor/generation](https://huggingface.co/docs/transformers/en/model_doc/qwen3_vl).
